# CS5224 Data Curation — Stage 1D

## Specification and pack-size parsing

Stage 1C safely normalized text, prices, stock and offer flags for all **177,638** source records while preserving the raw fields.

Stage 1D focuses only on `specification_raw`.

The goal is to convert specification strings conservatively into structured fields that can later support exact product matching:

- `specification_normalized`
- `pack_count`
- `unit_size`
- `unit`
- `total_quantity`
- `total_quantity_base`
- `base_unit`
- `spec_parse_status`
- `spec_parse_pattern`
- `spec_parse_notes`

### Examples

`6 × 100 g`

becomes:

- `pack_count = 6`
- `unit_size = 100`
- `unit = g`
- `total_quantity = 600`
- `base_unit = g`

`1.6 kg`

becomes:

- `pack_count = 1`
- `unit_size = 1.6`
- `unit = kg`
- `total_quantity = 1.6`
- `total_quantity_base = 1600`
- `base_unit = g`

`10 Per Pack`

becomes:

- `pack_count = 10`
- `unit_size = 1`
- `unit = piece`
- `total_quantity = 10`

`170g (12 per pack)`

is treated as **PARTIAL** rather than guessing an individual item weight.

### Parse statuses

- `PARSED` — the structure is sufficiently clear.
- `PARTIAL` — useful information is present, but some interpretation remains uncertain.
- `AMBIGUOUS` — the specification is not safely understood by the current rules.
- `MISSING` — the source specification is missing or blank.

Stage 1D does **not** infer product identity, match products across retailers, calculate promotions, or delete uncertain records.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd

# Notebook location:
# CartOptix/CartOptix/scripts

PROJECT_ROOT = Path.cwd().parent
STAGING_DATA_DIR = PROJECT_ROOT / "data" / "staging"

STAGE1C_FILE = (
    STAGING_DATA_DIR
    / "all_retailers_stage1c_normalized.csv"
)

print("Current notebook folder:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nStaging data directory:")
print(STAGING_DATA_DIR)

print("\nStage 1C input:")
print(STAGE1C_FILE)

print("\nExists:", STAGE1C_FILE.exists())

assert STAGE1C_FILE.exists(), (
    "Stage 1C checkpoint not found. "
    "Run Data_Curation_Stage1C.ipynb first."
)


Current notebook folder:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\scripts

Project root:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix

Staging data directory:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging

Stage 1C input:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\all_retailers_stage1c_normalized.csv

Exists: True


## 2. Define the expected Stage 1C schema

In [2]:
STAGE1C_COLUMNS = [
    "retailer",
    "source_file",
    "source_sheet",
    "source_row_number",
    "source_record_id",
    "source_category_raw",
    "source_category_normalized",
    "product_name_raw",
    "product_name_normalized",
    "brand_raw",
    "brand_normalized",
    "specification_raw",
    "product_price_raw",
    "product_price",
    "original_price_raw",
    "original_price",
    "savings_raw",
    "stock_raw",
    "in_stock",
    "offer_raw",
    "has_offer",
    "image_url_raw",
    "product_url_raw",
    "comments_raw",
]

EXPECTED_TOTAL_ROWS = 177_638

EXPECTED_RETAILER_COUNTS = {
    "redmart": 97_779,
    "fairprice": 69_402,
    "coldstorage": 10_457,
}


## 3. Load the Stage 1C checkpoint

In [3]:
stage1c = pd.read_csv(
    STAGE1C_FILE,
    dtype={
        "retailer": "string",
        "source_file": "string",
        "source_sheet": "string",
        "source_record_id": "string",
        "source_category_raw": "string",
        "source_category_normalized": "string",
        "product_name_raw": "string",
        "product_name_normalized": "string",
        "brand_raw": "string",
        "brand_normalized": "string",
        "specification_raw": "string",
        "stock_raw": "string",
        "offer_raw": "string",
        "image_url_raw": "string",
        "product_url_raw": "string",
        "comments_raw": "string",
    }
)

print("Loaded Stage 1C shape:")
print(stage1c.shape)

display(stage1c.head(3))


Loaded Stage 1C shape:
(177638, 24)


,retailer,source_file,source_sheet,source_row_number,source_record_id,source_category_raw,source_category_normalized,product_name_raw,product_name_normalized,brand_raw,...,original_price_raw,original_price,savings_raw,stock_raw,in_stock,offer_raw,has_offer,image_url_raw,product_url_raw,comments_raw
0,redmart,redmart_all_products_final.xlsx,Sheet1,2,redmart_00000002,shop-chilled-desserts,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,sojasun natural soya yoghurt,Sojasun,...,NaN,NaN,NaN,Yes,True,No,False,https://sg-live-01.slatic.net/p/5eaa6087b6c250...,https://www.lazada.sg/products/pdp-i303336401-...,<NA>
1,redmart,redmart_all_products_final.xlsx,Sheet1,3,redmart_00000003,shop-chilled-desserts,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix chocolate custard dessert cream 4x125g,Monoprix,...,NaN,NaN,Spend $45 + Free Gift,Yes,True,Yes,True,https://sg-test-11.slatic.net/p/9f86cfbce0b472...,https://www.lazada.sg/products/pdp-i462726209-...,<NA>
2,redmart,redmart_all_products_final.xlsx,Sheet1,4,redmart_00000004,shop-chilled-desserts,shop-chilled-desserts,Top Gourmet Cheng Tng,top gourmet cheng tng,Top Gourmet,...,NaN,NaN,NaN,Yes,True,No,False,https://sg-live-01.slatic.net/p/7579bbca2beaa0...,https://www.lazada.sg/products/pdp-i303250731-...,<NA>


## 4. Validate the Stage 1C input

In [4]:
actual_columns = stage1c.columns.tolist()

assert actual_columns == STAGE1C_COLUMNS, (
    "Stage 1C columns do not match the expected schema.\n"
    f"Expected: {STAGE1C_COLUMNS}\n"
    f"Actual:   {actual_columns}"
)

assert len(stage1c) == EXPECTED_TOTAL_ROWS

assert (
    stage1c["source_record_id"]
    .duplicated()
    .sum()
    == 0
)

assert (
    stage1c["source_record_id"]
    .isna()
    .sum()
    == 0
)

for retailer, expected_count in EXPECTED_RETAILER_COUNTS.items():
    actual_count = (
        stage1c["retailer"] == retailer
    ).sum()

    assert actual_count == expected_count, (
        f"{retailer}: expected {expected_count:,}, "
        f"found {actual_count:,}."
    )

print("PASS: Stage 1C input validation succeeded.")


PASS: Stage 1C input validation succeeded.


## 5. Preserve the complete Stage 1C dataset

Stage 1D only adds new fields. This snapshot is used later to verify that none of the 24 Stage 1C columns were changed.


In [5]:
stage1c_snapshot = stage1c.copy(deep=True)

print(
    f"Stage 1C snapshot created: "
    f"{stage1c_snapshot.shape[0]:,} rows x "
    f"{stage1c_snapshot.shape[1]} columns"
)


Stage 1C snapshot created: 177,638 rows x 24 columns


## 6. Define supported units

Stage 1D uses conservative metric-unit aliases.

For cross-retailer comparison, metric quantities are additionally converted into base units:

- `mg`, `g`, `kg` → base unit `g`
- `ml`, `cl`, `l` → base unit `ml`
- item counts → base unit `piece`

The original unit is still retained in `unit`.


In [6]:
METRIC_UNIT_ALIASES = {
    # Mass
    "mg": "mg",
    "milligram": "mg",
    "milligrams": "mg",

    "g": "g",
    "gm": "g",
    "gms": "g",
    "gram": "g",
    "grams": "g",

    "kg": "kg",
    "kgs": "kg",
    "kilogram": "kg",
    "kilograms": "kg",

    # Volume
    "ml": "ml",
    "millilitre": "ml",
    "millilitres": "ml",
    "milliliter": "ml",
    "milliliters": "ml",

    "cl": "cl",
    "centilitre": "cl",
    "centilitres": "cl",
    "centiliter": "cl",
    "centiliters": "cl",

    "l": "l",
    "lt": "l",
    "ltr": "l",
    "litre": "l",
    "litres": "l",
    "liter": "l",
    "liters": "l",
}

COUNT_UNIT_ALIASES = {
    "pc": "piece",
    "pcs": "piece",
    "piece": "piece",
    "pieces": "piece",
    "ea": "piece",
    "each": "piece",
}

# Longer alternatives are placed first so regex matching is stable.
METRIC_UNIT_REGEX = "|".join(
    sorted(
        map(re.escape, METRIC_UNIT_ALIASES.keys()),
        key=len,
        reverse=True
    )
)

COUNT_UNIT_REGEX = "|".join(
    sorted(
        map(re.escape, COUNT_UNIT_ALIASES.keys()),
        key=len,
        reverse=True
    )
)

NUMBER_REGEX = r"(\d+(?:\.\d+)?)"


## 7. Normalize specification text conservatively

In [7]:
def normalize_specification(value):
    """
    Normalize a specification string without changing its meaning.

    Operations:
    - preserve missing values;
    - Unicode NFKC normalization;
    - convert multiplication sign × to x;
    - collapse repeated whitespace;
    - casefold for case-insensitive parsing.

    The original specification_raw remains unchanged.
    """

    if pd.isna(value):
        return pd.NA

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    text = text.replace("×", "x")

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    if text == "":
        return pd.NA

    return text.casefold()


## 8. Define base-unit conversion

In [8]:
def convert_to_base_quantity(quantity, unit):
    """
    Convert a quantity into a common comparison unit.

    Returns:
        (base_quantity, base_unit)
    """

    if (
        quantity is None
        or pd.isna(quantity)
        or unit is None
        or pd.isna(unit)
    ):
        return None, None

    conversion = {
        "mg": ("g", 0.001),
        "g": ("g", 1.0),
        "kg": ("g", 1000.0),

        "ml": ("ml", 1.0),
        "cl": ("ml", 10.0),
        "l": ("ml", 1000.0),

        "piece": ("piece", 1.0),
    }

    if unit not in conversion:
        return None, None

    base_unit, factor = conversion[unit]

    base_quantity = (
        float(quantity)
        * factor
    )

    return base_quantity, base_unit


## 9. Define the conservative specification parser

The parser only marks a record as `PARSED` when the complete normalized specification matches a supported pattern.

It intentionally does not search for a convenient number inside a complicated string and then pretend the whole specification has been understood.


In [9]:
def parse_specification(value):
    """
    Parse a retailer specification conservatively.

    Output fields:
      specification_normalized
      pack_count
      unit_size
      unit
      total_quantity
      total_quantity_base
      base_unit
      spec_parse_status
      spec_parse_pattern
      spec_parse_notes
    """

    result = {
        "specification_normalized": normalize_specification(value),
        "pack_count": None,
        "unit_size": None,
        "unit": None,
        "total_quantity": None,
        "total_quantity_base": None,
        "base_unit": None,
        "spec_parse_status": "MISSING",
        "spec_parse_pattern": "MISSING",
        "spec_parse_notes": None,
    }

    text = result["specification_normalized"]

    if pd.isna(text):
        return result

    # =========================================================
    # Pattern 1: clear metric multipack
    #
    # Examples:
    #   6 x 100 g
    #   4x125g
    #   2 x 1.5 l
    #   6 x 220ml
    # =========================================================

    multi_pattern = (
        rf"(\d+)\s*x\s*"
        rf"{NUMBER_REGEX}\s*"
        rf"({METRIC_UNIT_REGEX})"
    )

    match = re.fullmatch(
        multi_pattern,
        text
    )

    if match:
        pack_count = int(match.group(1))
        unit_size = float(match.group(2))

        unit = METRIC_UNIT_ALIASES[
            match.group(3)
        ]

        total_quantity = (
            pack_count
            * unit_size
        )

        (
            total_quantity_base,
            base_unit
        ) = convert_to_base_quantity(
            total_quantity,
            unit
        )

        result.update(
            {
                "pack_count": pack_count,
                "unit_size": unit_size,
                "unit": unit,
                "total_quantity": total_quantity,
                "total_quantity_base": total_quantity_base,
                "base_unit": base_unit,
                "spec_parse_status": "PARSED",
                "spec_parse_pattern": "MULTIPACK_METRIC",
            }
        )

        return result

    # =========================================================
    # Pattern 2: total metric quantity plus item count
    #
    # Example:
    #   170g (12 per pack)
    #   500 ml (6 pcs)
    #
    # This is PARTIAL because the total quantity and item count
    # are known, but individual item size is not inferred.
    # =========================================================

    total_plus_count_pattern = (
        rf"{NUMBER_REGEX}\s*"
        rf"({METRIC_UNIT_REGEX})"
        rf"\s*\(\s*"
        rf"(\d+)\s*"
        rf"(?:per\s*pack|{COUNT_UNIT_REGEX}|s)"
        rf"\s*\)"
    )

    match = re.fullmatch(
        total_plus_count_pattern,
        text
    )

    if match:
        total_quantity = float(
            match.group(1)
        )

        unit = METRIC_UNIT_ALIASES[
            match.group(2)
        ]

        pack_count = int(
            match.group(3)
        )

        (
            total_quantity_base,
            base_unit
        ) = convert_to_base_quantity(
            total_quantity,
            unit
        )

        result.update(
            {
                "pack_count": pack_count,
                "unit_size": None,
                "unit": unit,
                "total_quantity": total_quantity,
                "total_quantity_base": total_quantity_base,
                "base_unit": base_unit,
                "spec_parse_status": "PARTIAL",
                "spec_parse_pattern": "TOTAL_PLUS_COUNT",
                "spec_parse_notes": (
                    "Total quantity and item count identified; "
                    "individual unit size was not inferred."
                ),
            }
        )

        return result

    # =========================================================
    # Pattern 3: clear single metric quantity
    #
    # Examples:
    #   400 g
    #   1.6kg
    #   750 ml
    #   2 l
    # =========================================================

    single_metric_pattern = (
        rf"{NUMBER_REGEX}\s*"
        rf"({METRIC_UNIT_REGEX})"
    )

    match = re.fullmatch(
        single_metric_pattern,
        text
    )

    if match:
        unit_size = float(
            match.group(1)
        )

        unit = METRIC_UNIT_ALIASES[
            match.group(2)
        ]

        pack_count = 1
        total_quantity = unit_size

        (
            total_quantity_base,
            base_unit
        ) = convert_to_base_quantity(
            total_quantity,
            unit
        )

        result.update(
            {
                "pack_count": pack_count,
                "unit_size": unit_size,
                "unit": unit,
                "total_quantity": total_quantity,
                "total_quantity_base": total_quantity_base,
                "base_unit": base_unit,
                "spec_parse_status": "PARSED",
                "spec_parse_pattern": "SINGLE_METRIC",
            }
        )

        return result

    # =========================================================
    # Pattern 4: item count written as "N Per Pack"
    #
    # Example:
    #   10 Per Pack
    # =========================================================

    count_per_pack_pattern = (
        r"(\d+)\s*per\s*pack"
    )

    match = re.fullmatch(
        count_per_pack_pattern,
        text
    )

    if match:
        pack_count = int(
            match.group(1)
        )

        result.update(
            {
                "pack_count": pack_count,
                "unit_size": 1.0,
                "unit": "piece",
                "total_quantity": float(pack_count),
                "total_quantity_base": float(pack_count),
                "base_unit": "piece",
                "spec_parse_status": "PARSED",
                "spec_parse_pattern": "COUNT_PER_PACK",
            }
        )

        return result

    # =========================================================
    # Pattern 5: explicit count unit
    #
    # Examples:
    #   72 Pcs
    #   1 each
    #   6 pieces
    # =========================================================

    count_piece_pattern = (
        rf"(\d+)\s*"
        rf"({COUNT_UNIT_REGEX})"
    )

    match = re.fullmatch(
        count_piece_pattern,
        text
    )

    if match:
        pack_count = int(
            match.group(1)
        )

        result.update(
            {
                "pack_count": pack_count,
                "unit_size": 1.0,
                "unit": "piece",
                "total_quantity": float(pack_count),
                "total_quantity_base": float(pack_count),
                "base_unit": "piece",
                "spec_parse_status": "PARSED",
                "spec_parse_pattern": "COUNT_PIECES",
            }
        )

        return result

    # =========================================================
    # Pattern 6: retail "s" notation
    #
    # Examples:
    #   6s
    #   12 s
    #
    # This is marked PARTIAL rather than PARSED because "s"
    # is retailer notation and should remain reviewable.
    # =========================================================

    count_s_pattern = (
        r"(\d+)\s*s"
    )

    match = re.fullmatch(
        count_s_pattern,
        text
    )

    if match:
        pack_count = int(
            match.group(1)
        )

        result.update(
            {
                "pack_count": pack_count,
                "unit_size": 1.0,
                "unit": "piece",
                "total_quantity": float(pack_count),
                "total_quantity_base": float(pack_count),
                "base_unit": "piece",
                "spec_parse_status": "PARTIAL",
                "spec_parse_pattern": "COUNT_S_SUFFIX",
                "spec_parse_notes": (
                    "Retail 's' notation interpreted as an item "
                    "count, but retained for manual review."
                ),
            }
        )

        return result

    # =========================================================
    # Anything else is deliberately not guessed.
    # =========================================================

    result.update(
        {
            "spec_parse_status": "AMBIGUOUS",
            "spec_parse_pattern": "UNRECOGNIZED",
            "spec_parse_notes": (
                "Specification was not matched by a conservative "
                "Stage 1D parsing rule."
            ),
        }
    )

    return result


## 10. Unit-test the parser before using the full dataset

These tests cover the main patterns Stage 1D is intended to understand.


In [10]:
TEST_CASES = [
    {
        "spec": "6 × 100 g",
        "status": "PARSED",
        "pattern": "MULTIPACK_METRIC",
        "pack_count": 6,
        "unit_size": 100.0,
        "unit": "g",
        "total_quantity": 600.0,
        "total_quantity_base": 600.0,
        "base_unit": "g",
    },
    {
        "spec": "4 x 125g",
        "status": "PARSED",
        "pattern": "MULTIPACK_METRIC",
        "pack_count": 4,
        "unit_size": 125.0,
        "unit": "g",
        "total_quantity": 500.0,
        "total_quantity_base": 500.0,
        "base_unit": "g",
    },
    {
        "spec": "2 x 1.5 L",
        "status": "PARSED",
        "pattern": "MULTIPACK_METRIC",
        "pack_count": 2,
        "unit_size": 1.5,
        "unit": "l",
        "total_quantity": 3.0,
        "total_quantity_base": 3000.0,
        "base_unit": "ml",
    },
    {
        "spec": "400 g",
        "status": "PARSED",
        "pattern": "SINGLE_METRIC",
        "pack_count": 1,
        "unit_size": 400.0,
        "unit": "g",
        "total_quantity": 400.0,
        "total_quantity_base": 400.0,
        "base_unit": "g",
    },
    {
        "spec": "1.6kg",
        "status": "PARSED",
        "pattern": "SINGLE_METRIC",
        "pack_count": 1,
        "unit_size": 1.6,
        "unit": "kg",
        "total_quantity": 1.6,
        "total_quantity_base": 1600.0,
        "base_unit": "g",
    },
    {
        "spec": "6 x 220ml",
        "status": "PARSED",
        "pattern": "MULTIPACK_METRIC",
        "pack_count": 6,
        "unit_size": 220.0,
        "unit": "ml",
        "total_quantity": 1320.0,
        "total_quantity_base": 1320.0,
        "base_unit": "ml",
    },
    {
        "spec": "10 Per Pack",
        "status": "PARSED",
        "pattern": "COUNT_PER_PACK",
        "pack_count": 10,
        "unit_size": 1.0,
        "unit": "piece",
        "total_quantity": 10.0,
        "total_quantity_base": 10.0,
        "base_unit": "piece",
    },
    {
        "spec": "72 Pcs",
        "status": "PARSED",
        "pattern": "COUNT_PIECES",
        "pack_count": 72,
        "unit_size": 1.0,
        "unit": "piece",
        "total_quantity": 72.0,
        "total_quantity_base": 72.0,
        "base_unit": "piece",
    },
    {
        "spec": "1each",
        "status": "PARSED",
        "pattern": "COUNT_PIECES",
        "pack_count": 1,
        "unit_size": 1.0,
        "unit": "piece",
        "total_quantity": 1.0,
        "total_quantity_base": 1.0,
        "base_unit": "piece",
    },
    {
        "spec": "170g (12 per pack)",
        "status": "PARTIAL",
        "pattern": "TOTAL_PLUS_COUNT",
        "pack_count": 12,
        "unit_size": None,
        "unit": "g",
        "total_quantity": 170.0,
        "total_quantity_base": 170.0,
        "base_unit": "g",
    },
    {
        "spec": "12s",
        "status": "PARTIAL",
        "pattern": "COUNT_S_SUFFIX",
        "pack_count": 12,
        "unit_size": 1.0,
        "unit": "piece",
        "total_quantity": 12.0,
        "total_quantity_base": 12.0,
        "base_unit": "piece",
    },
    {
        "spec": "assorted",
        "status": "AMBIGUOUS",
        "pattern": "UNRECOGNIZED",
        "pack_count": None,
        "unit_size": None,
        "unit": None,
        "total_quantity": None,
        "total_quantity_base": None,
        "base_unit": None,
    },
    {
        "spec": None,
        "status": "MISSING",
        "pattern": "MISSING",
        "pack_count": None,
        "unit_size": None,
        "unit": None,
        "total_quantity": None,
        "total_quantity_base": None,
        "base_unit": None,
    },
]

for test in TEST_CASES:
    parsed = parse_specification(
        test["spec"]
    )

    assert (
        parsed["spec_parse_status"]
        == test["status"]
    )

    assert (
        parsed["spec_parse_pattern"]
        == test["pattern"]
    )

    for field in [
        "pack_count",
        "unit_size",
        "unit",
        "total_quantity",
        "total_quantity_base",
        "base_unit",
    ]:
        expected = test[field]
        actual = parsed[field]

        if expected is None:
            assert (
                actual is None
                or pd.isna(actual)
            ), (
                f"{test['spec']}: {field} "
                f"expected missing, got {actual}"
            )

        elif isinstance(
            expected,
            float
        ):
            assert np.isclose(
                float(actual),
                expected
            ), (
                f"{test['spec']}: {field} "
                f"expected {expected}, got {actual}"
            )

        else:
            assert actual == expected

print(
    f"PASS: {len(TEST_CASES)} "
    "specification parser tests succeeded."
)


PASS: 13 specification parser tests succeeded.


## 11. Show the parser test results

In [11]:
test_results = []

for test in TEST_CASES:
    parsed = parse_specification(
        test["spec"]
    )

    test_results.append(
        {
            "specification_raw": test["spec"],
            **parsed,
        }
    )

display(
    pd.DataFrame(test_results)
)


,specification_raw,specification_normalized,pack_count,unit_size,unit,total_quantity,total_quantity_base,base_unit,spec_parse_status,spec_parse_pattern,spec_parse_notes
0,6 × 100 g,6 x 100 g,6.0,100.0,g,600.0,600.0,g,PARSED,MULTIPACK_METRIC,NaN
1,4 x 125g,4 x 125g,4.0,125.0,g,500.0,500.0,g,PARSED,MULTIPACK_METRIC,NaN
2,2 x 1.5 L,2 x 1.5 l,2.0,1.5,l,3.0,3000.0,ml,PARSED,MULTIPACK_METRIC,NaN
3,400 g,400 g,1.0,400.0,g,400.0,400.0,g,PARSED,SINGLE_METRIC,NaN
4,1.6kg,1.6kg,1.0,1.6,kg,1.6,1600.0,g,PARSED,SINGLE_METRIC,NaN
5,6 x 220ml,6 x 220ml,6.0,220.0,ml,1320.0,1320.0,ml,PARSED,MULTIPACK_METRIC,NaN
6,10 Per Pack,10 per pack,10.0,1.0,piece,10.0,10.0,piece,PARSED,COUNT_PER_PACK,NaN
7,72 Pcs,72 pcs,72.0,1.0,piece,72.0,72.0,piece,PARSED,COUNT_PIECES,NaN
8,1each,1each,1.0,1.0,piece,1.0,1.0,piece,PARSED,COUNT_PIECES,NaN
9,170g (12 per pack),170g (12 per pack),12.0,NaN,g,170.0,170.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...


## 12. Apply the parser to all Stage 1C records

In [12]:
parsed_specifications = pd.DataFrame(
    [
        parse_specification(value)
        for value in stage1c[
            "specification_raw"
        ]
    ],
    index=stage1c.index
)

print("Parsed specification table:")
print(parsed_specifications.shape)

display(parsed_specifications.head(10))


Parsed specification table:
(177638, 10)


,specification_normalized,pack_count,unit_size,unit,total_quantity,total_quantity_base,base_unit,spec_parse_status,spec_parse_pattern,spec_parse_notes
0,6 x 100 g,6.0,100.0,g,600.0,600.0,g,PARSED,MULTIPACK_METRIC,NaN
1,4 x 125 g,4.0,125.0,g,500.0,500.0,g,PARSED,MULTIPACK_METRIC,NaN
2,400 g,1.0,400.0,g,400.0,400.0,g,PARSED,SINGLE_METRIC,NaN
3,4 x 115 g,4.0,115.0,g,460.0,460.0,g,PARSED,MULTIPACK_METRIC,NaN
4,4 x 100 g,4.0,100.0,g,400.0,400.0,g,PARSED,MULTIPACK_METRIC,NaN
5,1 kg,1.0,1.0,kg,1.0,1000.0,g,PARSED,SINGLE_METRIC,NaN
6,430 g,1.0,430.0,g,430.0,430.0,g,PARSED,SINGLE_METRIC,NaN
7,200 g,1.0,200.0,g,200.0,200.0,g,PARSED,SINGLE_METRIC,NaN
8,360 g,1.0,360.0,g,360.0,360.0,g,PARSED,SINGLE_METRIC,NaN
9,200 g,1.0,200.0,g,200.0,200.0,g,PARSED,SINGLE_METRIC,NaN


## 13. Assign nullable data types

In [13]:
parsed_specifications[
    "specification_normalized"
] = parsed_specifications[
    "specification_normalized"
].astype("string")

parsed_specifications[
    "pack_count"
] = parsed_specifications[
    "pack_count"
].astype("Int64")

for column in [
    "unit_size",
    "total_quantity",
    "total_quantity_base",
]:
    parsed_specifications[
        column
    ] = parsed_specifications[
        column
    ].astype("Float64")

for column in [
    "unit",
    "base_unit",
    "spec_parse_status",
    "spec_parse_pattern",
    "spec_parse_notes",
]:
    parsed_specifications[
        column
    ] = parsed_specifications[
        column
    ].astype("string")

print(parsed_specifications.dtypes)


specification_normalized     string
pack_count                    Int64
unit_size                   Float64
unit                         string
total_quantity              Float64
total_quantity_base         Float64
base_unit                    string
spec_parse_status            string
spec_parse_pattern           string
spec_parse_notes             string
dtype: object


## 14. Attach Stage 1D fields to the Stage 1C dataset

In [14]:
stage1d = pd.concat(
    [
        stage1c.copy(),
        parsed_specifications,
    ],
    axis=1
)

print("Stage 1D shape:")
print(stage1d.shape)


Stage 1D shape:
(177638, 34)


Stage 1D should contain **34 columns**:

- 24 Stage 1C columns
- 10 Stage 1D specification fields


## 15. Put Stage 1D columns into a clear order

In [15]:
STAGE1D_COLUMNS = [
    # Provenance
    "retailer",
    "source_file",
    "source_sheet",
    "source_row_number",
    "source_record_id",

    # Category
    "source_category_raw",
    "source_category_normalized",

    # Product name
    "product_name_raw",
    "product_name_normalized",

    # Brand
    "brand_raw",
    "brand_normalized",

    # Specification / pack structure
    "specification_raw",
    "specification_normalized",
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
    "spec_parse_status",
    "spec_parse_pattern",
    "spec_parse_notes",

    # Prices
    "product_price_raw",
    "product_price",
    "original_price_raw",
    "original_price",

    # Promotion text
    "savings_raw",

    # Stock
    "stock_raw",
    "in_stock",

    # Offer
    "offer_raw",
    "has_offer",

    # Source URLs / comments
    "image_url_raw",
    "product_url_raw",
    "comments_raw",
]

stage1d = stage1d[
    STAGE1D_COLUMNS
]

print("Number of Stage 1D columns:")
print(len(stage1d.columns))

for i, column in enumerate(
    stage1d.columns,
    start=1
):
    print(f"{i:2d}. {column}")

assert len(stage1d.columns) == 34


Number of Stage 1D columns:
34
 1. retailer
 2. source_file
 3. source_sheet
 4. source_row_number
 5. source_record_id
 6. source_category_raw
 7. source_category_normalized
 8. product_name_raw
 9. product_name_normalized
10. brand_raw
11. brand_normalized
12. specification_raw
13. specification_normalized
14. pack_count
15. unit_size
16. unit
17. total_quantity
18. total_quantity_base
19. base_unit
20. spec_parse_status
21. spec_parse_pattern
22. spec_parse_notes
23. product_price_raw
24. product_price
25. original_price_raw
26. original_price
27. savings_raw
28. stock_raw
29. in_stock
30. offer_raw
31. has_offer
32. image_url_raw
33. product_url_raw
34. comments_raw


## 16. Verify Stage 1C fields were not changed

In [16]:
assert stage1d[
    STAGE1C_COLUMNS
].equals(stage1c_snapshot), (
    "At least one Stage 1C field changed during Stage 1D."
)

print(
    "PASS: All 24 Stage 1C fields remain unchanged."
)


PASS: All 24 Stage 1C fields remain unchanged.


## 17. Validate parse-status values

In [17]:
VALID_PARSE_STATUSES = {
    "PARSED",
    "PARTIAL",
    "AMBIGUOUS",
    "MISSING",
}

actual_statuses = set(
    stage1d[
        "spec_parse_status"
    ].dropna().unique()
)

unexpected_statuses = (
    actual_statuses
    - VALID_PARSE_STATUSES
)

print(
    "Parse statuses:",
    sorted(actual_statuses)
)

print(
    "Unexpected statuses:",
    sorted(unexpected_statuses)
)

assert not unexpected_statuses

print("\nPASS: All parse statuses are valid.")


Parse statuses: ['AMBIGUOUS', 'MISSING', 'PARSED', 'PARTIAL']
Unexpected statuses: []

PASS: All parse statuses are valid.


## 18. Show parse-status counts

In [18]:
parse_status_counts = (
    stage1d[
        "spec_parse_status"
    ]
    .value_counts(
        dropna=False
    )
)

print(parse_status_counts)

print("\nPercentages:")

print(
    (
        parse_status_counts
        / len(stage1d)
        * 100
    ).round(2)
)


spec_parse_status
PARSED       139918
PARTIAL       27662
MISSING        6915
AMBIGUOUS      3143
Name: count, dtype: Int64

Percentages:
spec_parse_status
PARSED       78.77
PARTIAL      15.57
MISSING       3.89
AMBIGUOUS     1.77
Name: count, dtype: Float64


## 19. Show parse-pattern counts

In [19]:
parse_pattern_counts = (
    stage1d[
        "spec_parse_pattern"
    ]
    .value_counts(
        dropna=False
    )
)

print(parse_pattern_counts)


spec_parse_pattern
SINGLE_METRIC       102981
COUNT_S_SUFFIX       27485
COUNT_PER_PACK       27090
MULTIPACK_METRIC      8536
MISSING               6915
UNRECOGNIZED          3143
COUNT_PIECES          1311
TOTAL_PLUS_COUNT       177
Name: count, dtype: Int64


In [20]:
nonpositive_mask = (
    (
        parsed_specifications["pack_count"].notna()
        & (parsed_specifications["pack_count"] <= 0)
    )
    |
    (
        parsed_specifications["unit_size"].notna()
        & (parsed_specifications["unit_size"] <= 0)
    )
    |
    (
        parsed_specifications["total_quantity"].notna()
        & (parsed_specifications["total_quantity"] <= 0)
    )
    |
    (
        parsed_specifications["total_quantity_base"].notna()
        & (parsed_specifications["total_quantity_base"] <= 0)
    )
)

print(
    "Specifications with non-positive parsed quantities:",
    int(nonpositive_mask.sum())
)

invalid_spec_records = pd.concat(
    [
        stage1c[
            [
                "retailer",
                "source_record_id",
                "product_name_raw",
                "specification_raw",
                "product_url_raw",
            ]
        ],
        parsed_specifications[
            [
                "pack_count",
                "unit_size",
                "unit",
                "total_quantity",
                "total_quantity_base",
                "base_unit",
                "spec_parse_status",
                "spec_parse_pattern",
            ]
        ],
    ],
    axis=1
).loc[nonpositive_mask]

display(invalid_spec_records)

Specifications with non-positive parsed quantities: 9


,retailer,source_record_id,product_name_raw,specification_raw,product_url_raw,pack_count,unit_size,unit,total_quantity,total_quantity_base,base_unit,spec_parse_status,spec_parse_pattern
102977,fairprice,fairprice_00005200,KITH Smokeless BBQ Grill Accessories - Extendi...,0 S,https://www.fairprice.com.sg/product/kith-smok...,0,1.0,piece,0.0,0.0,piece,PARTIAL,COUNT_S_SUFFIX
103202,fairprice,fairprice_00005425,Odette Pyramid Electric Kettle,1700 x 0 ML,https://www.fairprice.com.sg/product/odette-py...,1700,0.0,ml,0.0,0.0,ml,PARSED,MULTIPACK_METRIC
149080,fairprice,fairprice_00051303,Arcos Roasting Carving Fork Forged 177mm Riviera,0 S,https://www.fairprice.com.sg/product/arcos-roa...,0,1.0,piece,0.0,0.0,piece,PARTIAL,COUNT_S_SUFFIX
149088,fairprice,fairprice_00051311,Arcos Cheese Knife Forged 145mm Riviera,0 S,https://www.fairprice.com.sg/product/arcos-che...,0,1.0,piece,0.0,0.0,piece,PARTIAL,COUNT_S_SUFFIX
149097,fairprice,fairprice_00051320,Arcos Santoku Knife Forged 180mm Riviera,0 S,https://www.fairprice.com.sg/product/arcos-san...,0,1.0,piece,0.0,0.0,piece,PARTIAL,COUNT_S_SUFFIX
149154,fairprice,fairprice_00051377,Arcos Bread Knife Forged 200mm Riviera,0 S,https://www.fairprice.com.sg/product/arcos-bre...,0,1.0,piece,0.0,0.0,piece,PARTIAL,COUNT_S_SUFFIX
149183,fairprice,fairprice_00051406,Arcos Chef Knife Forged 150mm Riviera,0 S,https://www.fairprice.com.sg/product/arcos-che...,0,1.0,piece,0.0,0.0,piece,PARTIAL,COUNT_S_SUFFIX
149184,fairprice,fairprice_00051407,Arcos Santoku Knife Forged 140mm Riviera,0 S,https://www.fairprice.com.sg/product/arcos-san...,0,1.0,piece,0.0,0.0,piece,PARTIAL,COUNT_S_SUFFIX
153741,fairprice,fairprice_00055964,Dr. Brown's Flexees Ergonomic Teether - Green,0 S,https://www.fairprice.com.sg/product/dr-browns...,0,1.0,piece,0.0,0.0,piece,PARTIAL,COUNT_S_SUFFIX


In [21]:
# Reclassify specifications that produced zero or negative
# structured quantities.
#
# We preserve specification_raw unchanged and do NOT guess
# replacement values.

invalid_nonpositive_mask = (
    (
        parsed_specifications["pack_count"].notna()
        & (parsed_specifications["pack_count"] <= 0)
    )
    |
    (
        parsed_specifications["unit_size"].notna()
        & (parsed_specifications["unit_size"] <= 0)
    )
    |
    (
        parsed_specifications["total_quantity"].notna()
        & (parsed_specifications["total_quantity"] <= 0)
    )
    |
    (
        parsed_specifications["total_quantity_base"].notna()
        & (parsed_specifications["total_quantity_base"] <= 0)
    )
)

invalid_count = int(
    invalid_nonpositive_mask.sum()
)

print(
    "Reclassifying non-positive specifications:",
    invalid_count
)

# Clear structured values because they are not safe to use.
structured_columns = [
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
]

parsed_specifications.loc[
    invalid_nonpositive_mask,
    structured_columns
] = None

# Mark these for later review.
parsed_specifications.loc[
    invalid_nonpositive_mask,
    "spec_parse_status"
] = "AMBIGUOUS"

parsed_specifications.loc[
    invalid_nonpositive_mask,
    "spec_parse_pattern"
] = "INVALID_NONPOSITIVE"

parsed_specifications.loc[
    invalid_nonpositive_mask,
    "spec_parse_notes"
] = (
    "Specification matched a parsing rule but produced "
    "a zero or negative quantity. Structured values were "
    "cleared and the record was quarantined for review."
)

print(
    "PASS: Non-positive specifications were "
    "reclassified as AMBIGUOUS."
)

Reclassifying non-positive specifications: 9
PASS: Non-positive specifications were reclassified as AMBIGUOUS.


In [22]:
# Rebuild Stage 1D after correcting parsed_specifications.
#
# This is necessary because stage1d was previously created
# before the non-positive specifications were reclassified.

stage1d = pd.concat(
    [
        stage1c.copy(),
        parsed_specifications,
    ],
    axis=1
)

# Restore the intended Stage 1D column order.
stage1d = stage1d[
    STAGE1D_COLUMNS
]

print("Stage 1D rebuilt after specification corrections.")
print("Shape:", stage1d.shape)

print(
    "INVALID_NONPOSITIVE records:",
    (
        stage1d["spec_parse_pattern"]
        == "INVALID_NONPOSITIVE"
    ).sum()
)

Stage 1D rebuilt after specification corrections.
Shape: (177638, 34)
INVALID_NONPOSITIVE records: 9


## 20. Validate numeric fields

In [23]:
invalid_pack_count = (
    stage1d["pack_count"].notna()
    & (stage1d["pack_count"] <= 0)
)

invalid_unit_size = (
    stage1d["unit_size"].notna()
    & (stage1d["unit_size"] <= 0)
)

invalid_total_quantity = (
    stage1d["total_quantity"].notna()
    & (stage1d["total_quantity"] <= 0)
)

invalid_base_quantity = (
    stage1d["total_quantity_base"].notna()
    & (stage1d["total_quantity_base"] <= 0)
)

print(
    "Invalid pack counts:",
    int(invalid_pack_count.sum())
)

print(
    "Invalid unit sizes:",
    int(invalid_unit_size.sum())
)

print(
    "Invalid total quantities:",
    int(invalid_total_quantity.sum())
)

print(
    "Invalid base quantities:",
    int(invalid_base_quantity.sum())
)

assert invalid_pack_count.sum() == 0
assert invalid_unit_size.sum() == 0
assert invalid_total_quantity.sum() == 0
assert invalid_base_quantity.sum() == 0

print(
    "\nPASS: Parsed numeric quantities are positive."
)


Invalid pack counts: 0
Invalid unit sizes: 0
Invalid total quantities: 0
Invalid base quantities: 0

PASS: Parsed numeric quantities are positive.


## 21. Validate records marked PARSED

In [24]:
parsed_mask = (
    stage1d[
        "spec_parse_status"
    ] == "PARSED"
)

parsed_rows = stage1d.loc[
    parsed_mask
]

required_for_parsed = [
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
]

missing_required_parsed = (
    parsed_rows[
        required_for_parsed
    ]
    .isna()
    .sum()
)

print(
    "Missing required fields among PARSED records:"
)

print(missing_required_parsed)

assert (
    missing_required_parsed.sum()
    == 0
)

print(
    "\nPASS: Every PARSED record has "
    "the required structured fields."
)


Missing required fields among PARSED records:
pack_count             0
unit_size              0
unit                   0
total_quantity         0
total_quantity_base    0
base_unit              0
dtype: int64

PASS: Every PARSED record has the required structured fields.


## 22. Validate multipack arithmetic

In [25]:
multipack_mask = (
    stage1d[
        "spec_parse_pattern"
    ] == "MULTIPACK_METRIC"
)

multipack_rows = stage1d.loc[
    multipack_mask
]

if len(multipack_rows) > 0:
    calculated_total = (
        multipack_rows["pack_count"]
        .astype(float)
        * multipack_rows["unit_size"]
        .astype(float)
    )

    multipack_consistent = np.isclose(
        calculated_total,
        multipack_rows[
            "total_quantity"
        ].astype(float)
    )

    assert multipack_consistent.all()

print(
    f"PASS: {len(multipack_rows):,} "
    "MULTIPACK_METRIC records have consistent arithmetic."
)


PASS: 8,535 MULTIPACK_METRIC records have consistent arithmetic.


## 23. Validate single-quantity arithmetic

In [26]:
single_mask = (
    stage1d[
        "spec_parse_pattern"
    ] == "SINGLE_METRIC"
)

single_rows = stage1d.loc[
    single_mask
]

if len(single_rows) > 0:
    assert (
        single_rows[
            "pack_count"
        ] == 1
    ).all()

    single_consistent = np.isclose(
        single_rows[
            "unit_size"
        ].astype(float),
        single_rows[
            "total_quantity"
        ].astype(float)
    )

    assert single_consistent.all()

print(
    f"PASS: {len(single_rows):,} "
    "SINGLE_METRIC records have consistent arithmetic."
)


PASS: 102,981 SINGLE_METRIC records have consistent arithmetic.


## 24. Validate base-unit conversion

In [27]:
conversion_check_rows = stage1d.loc[
    stage1d[
        "total_quantity"
    ].notna()
    & stage1d[
        "unit"
    ].notna()
].copy()

conversion_errors = []

for row in conversion_check_rows.itertuples():
    expected_quantity, expected_unit = (
        convert_to_base_quantity(
            row.total_quantity,
            row.unit
        )
    )

    if (
        expected_quantity is None
        or expected_unit is None
    ):
        conversion_errors.append(
            row.source_record_id
        )
        continue

    if (
        not np.isclose(
            float(row.total_quantity_base),
            float(expected_quantity)
        )
        or row.base_unit != expected_unit
    ):
        conversion_errors.append(
            row.source_record_id
        )

print(
    "Base-unit conversion errors:",
    len(conversion_errors)
)

assert len(conversion_errors) == 0

print(
    "PASS: Base-unit conversions are internally consistent."
)


Base-unit conversion errors: 0
PASS: Base-unit conversions are internally consistent.


## 25. Verify AMBIGUOUS records were not guessed

In [28]:
ambiguous_mask = (
    stage1d[
        "spec_parse_status"
    ] == "AMBIGUOUS"
)

ambiguous_structured_fields = [
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
]

ambiguous_non_null = (
    stage1d.loc[
        ambiguous_mask,
        ambiguous_structured_fields
    ]
    .notna()
    .sum()
    .sum()
)

print(
    "Structured values assigned to AMBIGUOUS records:",
    int(ambiguous_non_null)
)

assert ambiguous_non_null == 0

print(
    "PASS: AMBIGUOUS specifications were not guessed."
)


Structured values assigned to AMBIGUOUS records: 0
PASS: AMBIGUOUS specifications were not guessed.


## 26. Verify MISSING records

In [29]:
missing_mask = (
    stage1d[
        "spec_parse_status"
    ] == "MISSING"
)

missing_with_raw_value = (
    stage1d.loc[
        missing_mask,
        "specification_raw"
    ]
    .notna()
    .sum()
)

print(
    "MISSING records with non-null specification_raw:",
    int(missing_with_raw_value)
)

# A whitespace-only source string could become MISSING,
# so do not assert this must always be zero.

print(
    "MISSING specifications:",
    int(missing_mask.sum())
)


MISSING records with non-null specification_raw: 0
MISSING specifications: 6915


## 27. Inspect parsed examples

In [30]:
display_columns = [
    "retailer",
    "source_record_id",
    "product_name_raw",
    "specification_raw",
    "specification_normalized",
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
    "spec_parse_status",
    "spec_parse_pattern",
]

display(
    stage1d.loc[
        stage1d[
            "spec_parse_status"
        ] == "PARSED",
        display_columns
    ].head(20)
)


,retailer,source_record_id,product_name_raw,specification_raw,specification_normalized,pack_count,unit_size,unit,total_quantity,total_quantity_base,base_unit,spec_parse_status,spec_parse_pattern
0,redmart,redmart_00000002,Sojasun Natural Soya Yoghurt,6 × 100 g,6 x 100 g,6,100.0,g,600.0,600.0,g,PARSED,MULTIPACK_METRIC
1,redmart,redmart_00000003,Monoprix Chocolate Custard Dessert Cream 4x125G,4 × 125 g,4 x 125 g,4,125.0,g,500.0,500.0,g,PARSED,MULTIPACK_METRIC
2,redmart,redmart_00000004,Top Gourmet Cheng Tng,400 g,400 g,1,400.0,g,400.0,400.0,g,PARSED,SINGLE_METRIC
3,redmart,redmart_00000005,La Laitiere Vanilla Rice Pudding Riz Au Lait 4...,4 × 115 g,4 x 115 g,4,115.0,g,460.0,460.0,g,PARSED,MULTIPACK_METRIC
4,redmart,redmart_00000006,Monoprix Flan Caramel Vanilla Pudding 4X100G,4 × 100 g,4 x 100 g,4,100.0,g,400.0,400.0,g,PARSED,MULTIPACK_METRIC
5,redmart,redmart_00000007,Fortune Sweetened Chin Chow 1KG,1 kg,1 kg,1,1.0,kg,1.0,1000.0,g,PARSED,SINGLE_METRIC
6,redmart,redmart_00000008,RedMart Trio Swiss Roll (Singapore Special),430 g,430 g,1,430.0,g,430.0,430.0,g,PARSED,SINGLE_METRIC
7,redmart,redmart_00000009,Arla Protein Chocolate Pudding 200G - Lactose ...,200 g,200 g,1,200.0,g,200.0,200.0,g,PARSED,SINGLE_METRIC
8,redmart,redmart_00000010,Fortune Soya Beancurd Almond 360G,360 g,360 g,1,360.0,g,360.0,360.0,g,PARSED,SINGLE_METRIC
9,redmart,redmart_00000011,Arla Protein Hazelnut Pudding 200G - Lactose Free,200 g,200 g,1,200.0,g,200.0,200.0,g,PARSED,SINGLE_METRIC


## 28. Inspect PARTIAL examples

In [31]:
display(
    stage1d.loc[
        stage1d[
            "spec_parse_status"
        ] == "PARTIAL",
        display_columns
        + ["spec_parse_notes"]
    ].head(20)
)


,retailer,source_record_id,product_name_raw,specification_raw,specification_normalized,pack_count,unit_size,unit,total_quantity,total_quantity_base,base_unit,spec_parse_status,spec_parse_pattern,spec_parse_notes
97780,fairprice,fairprice_00000003,Sun-Maid Organic Raisins - Mini-Snacks,170g (12 per pack),170g (12 per pack),12,<NA>,g,170.0,170.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...
97812,fairprice,fairprice_00000035,Nature's Nutrition Organic SOD Rooibos Tea,100 S,100 s,100,1.0,piece,100.0,100.0,piece,PARTIAL,COUNT_S_SUFFIX,Retail 's' notation interpreted as an item cou...
97823,fairprice,fairprice_00000046,English Tea Shop Organic Tea - Chamomile Lavender,30g (20 per pack),30g (20 per pack),20,<NA>,g,30.0,30.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...
97984,fairprice,fairprice_00000207,FairPrice Frozen Roti Prata,480g (6 per pack),480g (6 per pack),6,<NA>,g,480.0,480.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...
97985,fairprice,fairprice_00000208,Fairprice Wraps - Wholemeal,360g (8 per pack),360g (8 per pack),8,<NA>,g,360.0,360.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...
97986,fairprice,fairprice_00000209,Mission Wraps - Wholegrain,360g (8 per pack),360g (8 per pack),8,<NA>,g,360.0,360.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...
97987,fairprice,fairprice_00000210,Mission Wraps - Original,360g (8 per pack),360g (8 per pack),8,<NA>,g,360.0,360.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...
97991,fairprice,fairprice_00000214,Mission Pizza Crust - Wholemeal,300g (2 per pack),300g (2 per pack),2,<NA>,g,300.0,300.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...
97995,fairprice,fairprice_00000218,Mission Pita - Wholemeal,400g (5 per pack),400g (5 per pack),5,<NA>,g,400.0,400.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...
97996,fairprice,fairprice_00000219,Mission Pita - Plain,400g (5 per pack),400g (5 per pack),5,<NA>,g,400.0,400.0,g,PARTIAL,TOTAL_PLUS_COUNT,Total quantity and item count identified; indi...


## 29. Inspect AMBIGUOUS examples

In [32]:
display(
    stage1d.loc[
        stage1d[
            "spec_parse_status"
        ] == "AMBIGUOUS",
        [
            "retailer",
            "source_record_id",
            "product_name_raw",
            "specification_raw",
            "specification_normalized",
            "spec_parse_status",
            "spec_parse_pattern",
            "spec_parse_notes",
        ]
    ].head(30)
)


,retailer,source_record_id,product_name_raw,specification_raw,specification_normalized,spec_parse_status,spec_parse_pattern,spec_parse_notes
113,redmart,redmart_00000115,The Whole Kitchen Gluten Free Energy Balls/ Pr...,2 × 12 Per Pack,2 x 12 per pack,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...
266,redmart,redmart_00000268,NOMIEO Adult Diaper - Pants L [Carton Deal],8 × 8 Pcs,8 x 8 pcs,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...
275,redmart,redmart_00000277,Sensi Adult Diapers Size XL - Case,12 × 8 Pcs,12 x 8 pcs,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...
335,redmart,redmart_00000337,Sensi Adult Diapers Size S - Case,12 × 12 Pcs,12 x 12 pcs,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...
349,redmart,redmart_00000351,NOMIEO Adult Diaper - Tape M [Carton Deal],8 × 8 Per Pack,8 x 8 per pack,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...
380,redmart,redmart_00000382,Unicharm - Lifree Rehabilitation Thin Light Pa...,3693.6 Pcs,3693.6 pcs,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...
386,redmart,redmart_00000388,Unicharm - Lifree Rehabilitation Thin Light Pa...,3488.4 Pcs,3488.4 pcs,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...
387,redmart,redmart_00000389,Unicharm - Lifree Rehabilitation Zero Leaks Th...,5198.4 Pcs,5198.4 pcs,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...
407,redmart,redmart_00000409,NOMIEO Adult Underpad [Carton Deal],12 × 12 Per Pack,12 x 12 per pack,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...
409,redmart,redmart_00000411,NOMIEO Adult Diaper - Tape L [Carton Deal],8 × 8 Per Pack,8 x 8 per pack,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...


## 30. Create a specification parsing report

This report shows how many records matched each rule for each retailer.


In [33]:
specification_report = (
    stage1d
    .groupby(
        [
            "retailer",
            "spec_parse_status",
            "spec_parse_pattern",
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="record_count"
    )
)

retailer_totals = (
    stage1d
    .groupby("retailer")
    .size()
    .rename("retailer_total")
    .reset_index()
)

specification_report = (
    specification_report
    .merge(
        retailer_totals,
        on="retailer",
        how="left"
    )
)

specification_report[
    "percentage_of_retailer"
] = (
    specification_report[
        "record_count"
    ]
    / specification_report[
        "retailer_total"
    ]
    * 100
).round(2)

display(specification_report)


,retailer,spec_parse_status,spec_parse_pattern,record_count,retailer_total,percentage_of_retailer
0,coldstorage,AMBIGUOUS,UNRECOGNIZED,730,10457,6.98
1,coldstorage,PARSED,COUNT_PIECES,52,10457,0.50
2,coldstorage,PARSED,MULTIPACK_METRIC,899,10457,8.60
3,coldstorage,PARSED,SINGLE_METRIC,8252,10457,78.91
4,coldstorage,PARTIAL,COUNT_S_SUFFIX,524,10457,5.01
5,fairprice,AMBIGUOUS,INVALID_NONPOSITIVE,9,69402,0.01
6,fairprice,AMBIGUOUS,UNRECOGNIZED,1635,69402,2.36
7,fairprice,MISSING,MISSING,234,69402,0.34
8,fairprice,PARSED,COUNT_PER_PACK,2827,69402,4.07
9,fairprice,PARSED,COUNT_PIECES,1183,69402,1.70


## 31. Create the manual-review queue

Every `PARTIAL` or `AMBIGUOUS` record is retained in a separate review file.

This is important because Stage 1D is intentionally conservative: uncertain specifications should be reviewed rather than silently guessed.


In [34]:
review_mask = (
    stage1d[
        "spec_parse_status"
    ].isin(
        [
            "PARTIAL",
            "AMBIGUOUS",
        ]
    )
)

specification_review = stage1d.loc[
    review_mask,
    [
        "retailer",
        "source_record_id",
        "source_category_raw",
        "product_name_raw",
        "brand_raw",
        "specification_raw",
        "specification_normalized",
        "pack_count",
        "unit_size",
        "unit",
        "total_quantity",
        "total_quantity_base",
        "base_unit",
        "spec_parse_status",
        "spec_parse_pattern",
        "spec_parse_notes",
        "product_url_raw",
    ]
].copy()

print(
    "Records requiring specification review:",
    f"{len(specification_review):,}"
)

display(
    specification_review.head(20)
)


Records requiring specification review: 30,806


,retailer,source_record_id,source_category_raw,product_name_raw,brand_raw,specification_raw,specification_normalized,pack_count,unit_size,unit,total_quantity,total_quantity_base,base_unit,spec_parse_status,spec_parse_pattern,spec_parse_notes,product_url_raw
113,redmart,redmart_00000115,shop-chilled-desserts,The Whole Kitchen Gluten Free Energy Balls/ Pr...,Whole Kitchen,2 × 12 Per Pack,2 x 12 per pack,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i305124899-...
266,redmart,redmart_00000268,Senior Health,NOMIEO Adult Diaper - Pants L [Carton Deal],NOMIEO,8 × 8 Pcs,8 x 8 pcs,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i2278786813...
275,redmart,redmart_00000277,Senior Health,Sensi Adult Diapers Size XL - Case,Sensi,12 × 8 Pcs,12 x 8 pcs,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i303450589-...
335,redmart,redmart_00000337,Senior Health,Sensi Adult Diapers Size S - Case,Sensi,12 × 12 Pcs,12 x 12 pcs,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i303448626-...
349,redmart,redmart_00000351,Senior Health,NOMIEO Adult Diaper - Tape M [Carton Deal],NOMIEO,8 × 8 Per Pack,8 x 8 per pack,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i2278890147...
380,redmart,redmart_00000382,Senior Health,Unicharm - Lifree Rehabilitation Thin Light Pa...,Unicharm,3693.6 Pcs,3693.6 pcs,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i2883176865...
386,redmart,redmart_00000388,Senior Health,Unicharm - Lifree Rehabilitation Thin Light Pa...,Unicharm,3488.4 Pcs,3488.4 pcs,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i2883152420...
387,redmart,redmart_00000389,Senior Health,Unicharm - Lifree Rehabilitation Zero Leaks Th...,Unicharm,5198.4 Pcs,5198.4 pcs,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i2883221126...
407,redmart,redmart_00000409,Senior Health,NOMIEO Adult Underpad [Carton Deal],NOMIEO,12 × 12 Per Pack,12 x 12 per pack,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i2278831566...
409,redmart,redmart_00000411,Senior Health,NOMIEO Adult Diaper - Tape L [Carton Deal],NOMIEO,8 × 8 Per Pack,8 x 8 per pack,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,AMBIGUOUS,UNRECOGNIZED,Specification was not matched by a conservativ...,https://www.lazada.sg/products/pdp-i2278820552...


## 32. Verify row counts and provenance remain unchanged

In [35]:
assert len(stage1d) == EXPECTED_TOTAL_ROWS

assert (
    stage1d[
        "source_record_id"
    ].nunique()
    == EXPECTED_TOTAL_ROWS
)

assert (
    stage1d[
        "source_record_id"
    ].duplicated().sum()
    == 0
)

for retailer, expected_count in EXPECTED_RETAILER_COUNTS.items():
    actual_count = (
        stage1d[
            "retailer"
        ] == retailer
    ).sum()

    assert actual_count == expected_count

print(
    "PASS: Stage 1D preserved all rows "
    "and provenance IDs."
)


PASS: Stage 1D preserved all rows and provenance IDs.


## 33. Save retailer-specific Stage 1D checkpoints

In [36]:
for retailer in [
    "redmart",
    "fairprice",
    "coldstorage",
]:
    retailer_df = (
        stage1d.loc[
            stage1d[
                "retailer"
            ] == retailer
        ]
        .copy()
    )

    output_file = (
        STAGING_DATA_DIR
        / f"{retailer}_stage1d_spec_parsed.csv"
    )

    retailer_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8"
    )

    print(
        f"Saved {retailer}: "
        f"{len(retailer_df):,} rows -> "
        f"{output_file}"
    )


Saved redmart: 97,779 rows -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\redmart_stage1d_spec_parsed.csv
Saved fairprice: 69,402 rows -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\fairprice_stage1d_spec_parsed.csv
Saved coldstorage: 10,457 rows -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\coldstorage_stage1d_spec_parsed.csv


## 34. Save the combined Stage 1D checkpoint

In [37]:
STAGE1D_OUTPUT_FILE = (
    STAGING_DATA_DIR
    / "all_retailers_stage1d_spec_parsed.csv"
)

stage1d.to_csv(
    STAGE1D_OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print(
    "Saved combined Stage 1D dataset:"
)

print(STAGE1D_OUTPUT_FILE)


Saved combined Stage 1D dataset:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\all_retailers_stage1d_spec_parsed.csv


## 35. Save the specification report and review queue

In [38]:
REPORT_OUTPUT_FILE = (
    STAGING_DATA_DIR
    / "stage1d_specification_report.csv"
)

REVIEW_OUTPUT_FILE = (
    STAGING_DATA_DIR
    / "stage1d_specification_review.csv"
)

specification_report.to_csv(
    REPORT_OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

specification_review.to_csv(
    REVIEW_OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print(
    "Saved specification report:"
)

print(REPORT_OUTPUT_FILE)

print(
    "\nSaved manual-review queue:"
)

print(REVIEW_OUTPUT_FILE)


Saved specification report:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1d_specification_report.csv

Saved manual-review queue:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\stage1d_specification_review.csv


## 36. Verify all Stage 1D output files exist

In [39]:
OUTPUT_FILES = [
    (
        STAGING_DATA_DIR
        / "redmart_stage1d_spec_parsed.csv"
    ),
    (
        STAGING_DATA_DIR
        / "fairprice_stage1d_spec_parsed.csv"
    ),
    (
        STAGING_DATA_DIR
        / "coldstorage_stage1d_spec_parsed.csv"
    ),
    STAGE1D_OUTPUT_FILE,
    REPORT_OUTPUT_FILE,
    REVIEW_OUTPUT_FILE,
]

for file_path in OUTPUT_FILES:
    print(
        f"{file_path.name:45s} -> "
        f"{file_path.exists()}"
    )

    assert file_path.exists()

print(
    "\nPASS: All Stage 1D output files were created."
)


redmart_stage1d_spec_parsed.csv               -> True
fairprice_stage1d_spec_parsed.csv             -> True
coldstorage_stage1d_spec_parsed.csv           -> True
all_retailers_stage1d_spec_parsed.csv         -> True
stage1d_specification_report.csv              -> True
stage1d_specification_review.csv              -> True

PASS: All Stage 1D output files were created.


## 37. Final Stage 1D validation

In [40]:
parsed_count = int(
    (
        stage1d[
            "spec_parse_status"
        ] == "PARSED"
    ).sum()
)

partial_count = int(
    (
        stage1d[
            "spec_parse_status"
        ] == "PARTIAL"
    ).sum()
)

ambiguous_count = int(
    (
        stage1d[
            "spec_parse_status"
        ] == "AMBIGUOUS"
    ).sum()
)

missing_count = int(
    (
        stage1d[
            "spec_parse_status"
        ] == "MISSING"
    ).sum()
)

print("=" * 76)
print("STAGE 1D FINAL VALIDATION")
print("=" * 76)

print(
    f"Total records              : "
    f"{len(stage1d):,}"
)

print(
    f"Unique source IDs          : "
    f"{stage1d['source_record_id'].nunique():,}"
)

print(
    f"Duplicate source IDs       : "
    f"{stage1d['source_record_id'].duplicated().sum():,}"
)

print(
    f"Number of columns          : "
    f"{len(stage1d.columns)}"
)

print("-" * 76)

print(
    f"PARSED specifications      : "
    f"{parsed_count:,}"
)

print(
    f"PARTIAL specifications     : "
    f"{partial_count:,}"
)

print(
    f"AMBIGUOUS specifications   : "
    f"{ambiguous_count:,}"
)

print(
    f"MISSING specifications     : "
    f"{missing_count:,}"
)

print("-" * 76)

print(
    f"Manual-review queue        : "
    f"{len(specification_review):,}"
)

print("=" * 76)

assert len(stage1d) == 177_638
assert stage1d["source_record_id"].nunique() == 177_638
assert stage1d["source_record_id"].duplicated().sum() == 0
assert len(stage1d.columns) == 34

assert (
    parsed_count
    + partial_count
    + ambiguous_count
    + missing_count
    == 177_638
)

assert stage1d[
    STAGE1C_COLUMNS
].equals(stage1c_snapshot)

print()
print(
    "PASS: Stage 1D specification and "
    "pack-size parsing succeeded."
)

print()
print(
    "Important: PARTIAL and AMBIGUOUS records "
    "have not been silently corrected. "
    "They are retained for review."
)


STAGE 1D FINAL VALIDATION
Total records              : 177,638
Unique source IDs          : 177,638
Duplicate source IDs       : 0
Number of columns          : 34
----------------------------------------------------------------------------
PARSED specifications      : 139,917
PARTIAL specifications     : 27,654
AMBIGUOUS specifications   : 3,152
MISSING specifications     : 6,915
----------------------------------------------------------------------------
Manual-review queue        : 30,806

PASS: Stage 1D specification and pack-size parsing succeeded.

Important: PARTIAL and AMBIGUOUS records have not been silently corrected. They are retained for review.


## Stage 1D result

When the final validation passes:

- all **177,638** Stage 1C records remain present;
- all source IDs remain unique;
- all Stage 1C values remain unchanged;
- clear metric single packs and multipacks are structured;
- clear item-count specifications are structured;
- metric quantities also have common base quantities for later matching;
- uncertain records are explicitly labelled `PARTIAL` or `AMBIGUOUS`;
- a manual-review queue is produced rather than guessing.

The next step after reviewing the Stage 1D results is normally **Stage 1E: category mapping and eligibility filtering**, before the cross-retailer pilot matching stage.
